# Community Detection on Yeast PPI Network\n\nThis notebook handles data loading, algorithm execution, ground-truth loading, and F-score evaluation.

In [1]:
\
!pip install --no-cache-dir cdlib leidenalg python-louvain networkx pandas scikit-learn matplotlib



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
\
import networkx as nx
import pandas as pd
from cdlib import algorithms, evaluation, NodeClustering
import os
import numpy as np
from collections import Counter
from itertools import combinations

# 1. Ensure the PPI graph loads correctly from dip_ppin.csv
data_path = "xgw/data/preprocessed/dip_ppin.csv"
print(f"Loading PPI network from {data_path}...")
df = pd.read_csv(data_path)
cols = df.columns
source_col, target_col = cols[0], cols[1]
ppi_graph = nx.from_pandas_edgelist(df, source=source_col, target=target_col)

print(f"Nodes: {len(ppi_graph.nodes)}")
print(f"Edges: {len(ppi_graph.edges)}")


Note: to be able to use all crisp methods, you need to install some additional packages:  {'infomap', 'graph_tool', 'wurlitzer', 'bayanpy'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'pyclustering', 'ASLPAw'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'infomap', 'wurlitzer'}
Loading PPI network from xgw/data/preprocessed/dip_ppin.csv...


Nodes: 4749
Edges: 22495


In [3]:
\
# 2. Run all community detection algorithms
methods = {}

print("Running Leiden...")
methods['Leiden'] = algorithms.leiden(ppi_graph)
print(f"  Found {len(methods['Leiden'].communities)} groups")

print("Running Louvain...")
methods['Louvain'] = algorithms.louvain(ppi_graph)
print(f"  Found {len(methods['Louvain'].communities)} groups")

print("Running Label Propagation...")
methods['Label Propagation'] = algorithms.label_propagation(ppi_graph)
print(f"  Found {len(methods['Label Propagation'].communities)} groups")

print("Running Clique Percolation method (k=3)...")
methods['Clique Percolation'] = algorithms.kclique(ppi_graph, k=3)
print(f"  Found {len(methods['Clique Percolation'].communities)} groups")

print("Running Girvan-Newman...")
print("  (Skipped due to computational intensity on ~5k nodes/22k edges)")
# methods['Girvan-Newman'] = algorithms.girvan_newman(ppi_graph, level=1)

print("Running Stochastic Block Model...")
try:
    methods['SBM'] = algorithms.sbm_dl(ppi_graph)
    print(f"  Found {len(methods['SBM'].communities)} groups")
except Exception as e:
    print("  SBM failed (graph-tool likely missing). Skipping SBM.")

print("Running Consensus clustering algorithm...")
# Optimized Consensus: Count co-occurrences using Counter
fast_comms = [methods['Leiden'].communities, methods['Louvain'].communities, methods['Label Propagation'].communities]
co_occurrence = Counter()

for comms in fast_comms:
    for c in comms:
        # Sort to ensure consistent edge keys
        c_list = sorted(list(c))
        co_occurrence.update(combinations(c_list, 2))

threshold = len(fast_comms) / 2.0
filtered_edges = [(u, v, count) for (u, v), count in co_occurrence.items() if count > threshold]

filtered_G = nx.Graph()
filtered_G.add_nodes_from(ppi_graph.nodes())
filtered_G.add_weighted_edges_from(filtered_edges)

consensus_clusters = algorithms.louvain(filtered_G)
methods['Consensus'] = NodeClustering(consensus_clusters.communities, ppi_graph, "Consensus")
print(f"  Found {len(methods['Consensus'].communities)} groups")


Running Leiden...
  Found 40 groups
Running Louvain...


  Found 46 groups
Running Label Propagation...
  Found 164 groups
Running Clique Percolation method (k=3)...


  Found 199 groups
Running Girvan-Newman...
  (Skipped due to computational intensity on ~5k nodes/22k edges)
Running Stochastic Block Model...
  SBM failed (graph-tool likely missing). Skipping SBM.
Running Consensus clustering algorithm...


  Found 41 groups


In [4]:
\
# 3. Load MCL reference clusters and CYC2008 ground-truth complexes
ground_truths = {}

def load_mcl_clusters(filepath, graph):
    clusters = []
    with open(filepath, 'r') as f:
        for line in f:
            members = line.strip().split('\t')
            valid_members = [m for m in members if m in graph]
            if len(valid_members) > 0:
                clusters.append(valid_members)
    return NodeClustering(clusters, graph, "MCL")

mcl_levels = [20, 30, 40, 50]
for lvl in mcl_levels:
    mcl_path = f"xgw/data/clusters/out.dip_unweighted.csv.I{lvl}"
    ground_truths[f'MCL I={lvl/10.0}'] = load_mcl_clusters(mcl_path, ppi_graph)
    print(f"MCL I={lvl/10.0} found {len(ground_truths[f'MCL I={lvl/10.0}'].communities)} groups")

cyc_path = "xgw/data/swc/complexes_CYC.txt"
cyc_df = pd.read_csv(cyc_path, sep='\t', header=None, names=['protein', 'complex_id', 'complex_name'])
cyc_clusters_raw = [group['protein'].tolist() for _, group in cyc_df.groupby('complex_id')]
cyc_clusters = [[m for m in c if m in ppi_graph] for c in cyc_clusters_raw]
cyc_clusters = [c for c in cyc_clusters if len(c) > 0]
ground_truths['CYC2008'] = NodeClustering(cyc_clusters, ppi_graph, "CYC2008")
print(f"CYC2008 found {len(ground_truths['CYC2008'].communities)} groups")


MCL I=2.0 found 265 groups
MCL I=3.0 found 561 groups
MCL I=4.0 found 751 groups
MCL I=5.0 found 877 groups
CYC2008 found 405 groups


In [5]:
\
# 4. Implement F-score matching evaluation
print("\n--- F-score Matching Evaluation ---")
for gt_name, gt_clustering in ground_truths.items():
    print(f"\nGround Truth: {gt_name}")
    for algo_name, algo_clustering in methods.items():
        try:
            f_score = evaluation.f1(algo_clustering, gt_clustering).score
            print(f"  {algo_name:20s}: {f_score:.4f}")
        except Exception as e:
            print(f"  {algo_name:20s}: Failed ({e})")



--- F-score Matching Evaluation ---

Ground Truth: MCL I=2.0
  Leiden              : 0.2411
  Louvain             : 0.2119
  Label Propagation   : 0.6259
  Clique Percolation  : 0.4421
  Consensus           : 0.2450

Ground Truth: MCL I=3.0
  Leiden              : 0.1823
  Louvain             : 0.1729
  Label Propagation   : 0.6403
  Clique Percolation  : 0.4619
  Consensus           : 0.2015

Ground Truth: MCL I=4.0
  Leiden              : 0.1588
  Louvain             : 0.1509
  Label Propagation   : 0.6376
  Clique Percolation  : 0.4747
  Consensus           : 0.2042

Ground Truth: MCL I=5.0
  Leiden              : 0.1586
  Louvain             : 0.1510
  Label Propagation   : 0.6388
  Clique Percolation  : 0.4848
  Consensus           : 0.1575

Ground Truth: CYC2008
  Leiden              : 0.1722
  Louvain             : 0.1514
  Label Propagation   : 0.4495


  Clique Percolation  : 0.4898


  Consensus           : 0.2018


In [6]:
\
# 5. Export GEXF files for Gephi with Python
import xml.etree.ElementTree as ET

os.makedirs('gephi_exports', exist_ok=True)

print("Computing spring layout for Gephi exports (iterations=20)...")
pos = nx.spring_layout(ppi_graph, k=0.15, iterations=20, seed=42)
scale = 1000
pos = {node: (x * scale, y * scale) for node, (x, y) in pos.items()}
print("Layout computed.")

def inject_positions_into_gexf(filepath, positions):
    tree = ET.parse(filepath)
    root = tree.getroot()
    ET.register_namespace('', 'http://www.gexf.net/1.2draft')
    ET.register_namespace('viz', 'http://www.gexf.net/1.2draft/viz')

    for attr in list(root.attrib):
        if 'viz' in attr and 'xmlns' in attr:
            del root.attrib[attr]

    for node_elem in root.iter('{http://www.gexf.net/1.2draft}node'):
        node_id = node_elem.get('id')
        if node_id in positions:
            x, y = positions[node_id]
            viz_pos = ET.SubElement(node_elem, '{http://www.gexf.net/1.2draft/viz}position')
            viz_pos.set('x', str(round(x, 2)))
            viz_pos.set('y', str(round(y, 2)))
            viz_pos.set('z', '0.0')

    tree.write(filepath, xml_declaration=True, encoding='utf-8')

for algo_name, clustering in methods.items():
    G = ppi_graph.copy()
    safe_name = algo_name.lower().replace(' ', '_').replace('-', '_')
    
    for comm_id, members in enumerate(clustering.communities):
        for node in members:
            if node in G.nodes:
                G.nodes[node]['community'] = comm_id
                
    for node in G.nodes:
        if 'community' not in G.nodes[node]:
            G.nodes[node]['community'] = -1
            
    out_path = f'gephi_exports/ppi_{safe_name}.gexf'
    nx.write_gexf(G, out_path)
    inject_positions_into_gexf(out_path, pos)
    print(f"Exported: {out_path}")

print("\nAll exports complete! Files are ready in the gephi_exports/ folder.")


Computing spring layout for Gephi exports (iterations=20)...


Layout computed.


Exported: gephi_exports/ppi_leiden.gexf


Exported: gephi_exports/ppi_louvain.gexf


Exported: gephi_exports/ppi_label_propagation.gexf


Exported: gephi_exports/ppi_clique_percolation.gexf


Exported: gephi_exports/ppi_consensus.gexf

All exports complete! Files are ready in the gephi_exports/ folder.
